# Longformer: Long Document Transformer

Longformer, introduced by Beltagy et al. (2020), addresses the challenge of scaling Transformers to long documents by introducing an efficient attention mechanism. Here, we'll delve into the mathematical details of this mechanism.

## 1. Standard Self-Attention

In the standard self-attention mechanism used by models like BERT, the attention score between two tokens is computed as follows:

$$
\text{Attention}(Q, K, V) = \text{softmax}\left(\frac{QK^T}{\sqrt{d_k}}\right) V
$$

where:
- $Q \in \mathbb{R}^{n \times d_k}$ is the query matrix,
- $K \in \mathbb{R}^{n \times d_k}$ is the key matrix,
- $V \in \mathbb{R}^{n \times d_v}$ is the value matrix,
- $n$ is the sequence length,
- $d_k$ and $d_v$ are the dimensions of the keys/queries and values, respectively.

The computational complexity of this operation is $O(n^2)$ due to the $QK^T$ matrix multiplication, which becomes prohibitive for long sequences.

## 2. Longformer's Attention Mechanism

Longformer introduces two main modifications: sliding window attention and global attention.

### Sliding Window Attention

Sliding window attention restricts each token to attend to a fixed-size local window around it. If the window size is $w$, each token attends to $w$ tokens instead of $n$.

Mathematically, for each token $i$:

$$
\text{Attention}(Q_i, K_i, V_i) = \text{softmax}\left(\frac{Q_i K_{i-w/2:i+w/2}^T}{\sqrt{d_k}}\right) V_{i-w/2:i+w/2}
$$

Here, $Q_i$, $K_{i-w/2:i+w/2}$, and $V_{i-w/2:i+w/2}$ are the query, key, and value vectors within the local window centered at $i$.

The computational complexity is reduced to $O(nw)$.

### Global Attention

Global attention allows a small subset of tokens (e.g., special tokens like [CLS]) to attend to all tokens in the sequence, facilitating information flow across the entire sequence.

Let $G$ be the set of global tokens. The attention for a global token $g \in G$ is computed as:

$$
\text{Attention}(Q_g, K, V) = \text{softmax}\left(\frac{Q_g K^T}{\sqrt{d_k}}\right) V
$$

The global tokens have a computational complexity of $O(|G|n)$.

## 3. Combined Attention

Combining both types of attention, the overall attention mechanism for each token is:

$$
\text{CombinedAttention}(Q, K, V) = \text{WindowedAttention}(Q, K, V) + \text{GlobalAttention}(Q, K, V)
$$

For tokens not in $G$:

$$
\text{Attention}_i = \text{softmax}\left(\frac{Q_i K_{i-w/2:i+w/2}^T}{\sqrt{d_k}}\right) V_{i-w/2:i+w/2}
$$

For tokens in $G$:

$$
\text{Attention}_g = \text{softmax}\left(\frac{Q_g K^T}{\sqrt{d_k}}\right) V
$$

## 4. Complexity Analysis

The combined complexity for a sequence with $n$ tokens, window size $w$, and $|G|$ global tokens is:

$$
O(nw + |G|n)
$$

For typical values $w \ll n$ and $|G| \ll n$, the complexity is much lower than the $O(n^2)$ complexity of the standard self-attention.

## References

1. Beltagy, I., Peters, M. E., & Cohan, A. (2020). Longformer: The Long-Document Transformer. arXiv preprint arXiv:2004.05150. Available at: [https://arxiv.org/abs/2004.05150](https://arxiv.org/abs/2004.05150)
2. Vaswani, A., Shazeer, N., Parmar, N., Uszkoreit, J., Jones, L., Gomez, A. N., ... & Polosukhin, I. (2017). Attention is All You Need. Advances in Neural Information Processing Systems, 30, 5998-6008. Available at: [https://arxiv.org/abs/1706.03762](https://arxiv.org/abs/1706.03762)

In [1]:
!pip install torch==2.3.1

  Using cached nvidia_cuda_nvrtc_cu12-12.1.105-py3-none-manylinux1_x86_64.whl.metadata (1.5 kB)
  Using cached nvidia_cuda_runtime_cu12-12.1.105-py3-none-manylinux1_x86_64.whl.metadata (1.5 kB)
  Using cached nvidia_cuda_cupti_cu12-12.1.105-py3-none-manylinux1_x86_64.whl.metadata (1.6 kB)
  Using cached nvidia_cudnn_cu12-8.9.2.26-py3-none-manylinux1_x86_64.whl.metadata (1.6 kB)
  Using cached nvidia_cublas_cu12-12.1.3.1-py3-none-manylinux1_x86_64.whl.metadata (1.5 kB)
  Using cached nvidia_cufft_cu12-11.0.2.54-py3-none-manylinux1_x86_64.whl.metadata (1.5 kB)
  Using cached nvidia_curand_cu12-10.3.2.106-py3-none-manylinux1_x86_64.whl.metadata (1.5 kB)
  Using cached nvidia_cusolver_cu12-11.4.5.107-py3-none-manylinux1_x86_64.whl.metadata (1.6 kB)
  Using cached nvidia_cusparse_cu12-12.1.0.106-py3-none-manylinux1_x86_64.whl.metadata (1.6 kB)
  Using cached nvidia_nccl_cu12-2.20.5-py3-none-manylinux2014_x86_64.whl.metadata (1.8 kB)
  Using cached nvidia_nvtx_cu12-12.1.105-py3-none-manylinu

In [3]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
import time

class StandardSelfAttention(nn.Module):
    def __init__(self, d_model, n_heads):
        super().__init__()
        self.mha = nn.MultiheadAttention(d_model, n_heads, batch_first=True)
        self.out = nn.Linear(d_model, d_model)

    def forward(self, x):
        attn_output, _ = self.mha(x, x, x)
        return self.out(attn_output)

class LongformerSelfAttention(nn.Module):
    def __init__(self, d_model, n_heads, window_size):
        super().__init__()
        self.d_model = d_model
        self.n_heads = n_heads
        self.head_dim = d_model // n_heads
        self.window_size = window_size

        self.q_linear = nn.Linear(d_model, d_model)
        self.k_linear = nn.Linear(d_model, d_model)
        self.v_linear = nn.Linear(d_model, d_model)
        self.out = nn.Linear(d_model, d_model)

    def forward(self, x):
        batch_size, seq_len, _ = x.shape

        q = self.q_linear(x).view(batch_size, seq_len, self.n_heads, self.head_dim).transpose(1, 2)
        k = self.k_linear(x).view(batch_size, seq_len, self.n_heads, self.head_dim).transpose(1, 2)
        v = self.v_linear(x).view(batch_size, seq_len, self.n_heads, self.head_dim).transpose(1, 2)

        padding = self.window_size // 2
        padded_k = nn.functional.pad(k, (0, 0, padding, padding))
        padded_v = nn.functional.pad(v, (0, 0, padding, padding))

        values = torch.zeros(batch_size, self.n_heads, seq_len, self.head_dim, device=x.device)

        for i in range(seq_len):
            start_idx = i
            end_idx = i + self.window_size
            local_k = padded_k[:, :, start_idx:end_idx]
            local_v = padded_v[:, :, start_idx:end_idx]

            scores = torch.matmul(q[:, :, i].unsqueeze(2), local_k.transpose(-2, -1)) / (self.head_dim ** 0.5)
            attn = torch.softmax(scores, dim=-1)
            values[:, :, i] = torch.matmul(attn, local_v).squeeze(2)

        context = values.transpose(1, 2).contiguous().view(batch_size, seq_len, self.d_model)
        return self.out(context)

class TransformerLayer(nn.Module):
    def __init__(self, d_model, n_heads, d_ff, attention_type='standard', window_size=None):
        super().__init__()
        if attention_type == 'standard':
            self.self_attention = StandardSelfAttention(d_model, n_heads)
        elif attention_type == 'longformer':
            self.self_attention = LongformerSelfAttention(d_model, n_heads, window_size)
        else:
            raise ValueError("Invalid attention type")

        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.feed_forward = nn.Sequential(
            nn.Linear(d_model, d_ff),
            nn.ReLU(),
            nn.Linear(d_ff, d_model)
        )

    def forward(self, x):
        x = x + self.self_attention(x)
        x = self.norm1(x)
        x = x + self.feed_forward(x)
        x = self.norm2(x)
        return x

class TransformerClassifier(nn.Module):
    def __init__(self, d_model, n_heads, d_ff, num_layers, attention_type='standard', window_size=None):
        super().__init__()
        self.layers = nn.ModuleList([
            TransformerLayer(d_model, n_heads, d_ff, attention_type, window_size)
            for _ in range(num_layers)
        ])
        self.fc = nn.Linear(d_model, 1)

    def forward(self, x):
        for layer in self.layers:
            x = layer(x)
        return self.fc(x.mean(dim=1))

def create_synthetic_dataset(batch_size, seq_len, d_model, num_samples):
    X = torch.randn(num_samples, seq_len, d_model)
    y = torch.sum(X, dim=(1,2)) > 0
    return TensorDataset(X, y.float())

@torch.no_grad()
def evaluate_model(model, data_loader, criterion, device):
    model.eval()
    total_loss = 0
    correct = 0
    total = 0
    for batch_x, batch_y in data_loader:
        batch_x, batch_y = batch_x.to(device), batch_y.to(device)
        output = model(batch_x).squeeze()
        loss = criterion(output, batch_y)
        total_loss += loss.item()
        predicted = (output > 0).float()
        total += batch_y.size(0)
        correct += (predicted == batch_y).sum().item()
    return total_loss / len(data_loader), correct / total

def train_model(model, train_loader, val_loader, criterion, optimizer, device, num_epochs):
    model.train()
    for epoch in range(num_epochs):
        total_loss = 0
        for batch_x, batch_y in train_loader:
            batch_x, batch_y = batch_x.to(device), batch_y.to(device)
            optimizer.zero_grad()
            output = model(batch_x).squeeze()
            loss = criterion(output, batch_y)
            loss.backward()
            optimizer.step()
            total_loss += loss.item()

        train_loss = total_loss / len(train_loader)
        val_loss, val_acc = evaluate_model(model, val_loader, criterion, device)
        print(f"Epoch {epoch+1}/{num_epochs}, Train Loss: {train_loss:.4f}, Val Loss: {val_loss:.4f}, Val Acc: {val_acc:.4f}")

def main():
    # Hyperparameters
    batch_size = 32
    seq_len = 512
    d_model = 64
    n_heads = 4
    d_ff = 256
    num_samples = 10000
    num_epochs = 10
    learning_rate = 0.001
    window_size = 64  # for Longformer
    num_layers = 2

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    # Create synthetic dataset
    full_dataset = create_synthetic_dataset(batch_size, seq_len, d_model, num_samples)
    train_size = int(0.8 * len(full_dataset))
    val_size = len(full_dataset) - train_size
    train_dataset, val_dataset = torch.utils.data.random_split(full_dataset, [train_size, val_size])
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
    val_loader = DataLoader(val_dataset, batch_size=batch_size)

    # Standard Transformer
    standard_model = TransformerClassifier(d_model, n_heads, d_ff, num_layers, attention_type='standard').to(device)
    standard_criterion = nn.BCEWithLogitsLoss()
    standard_optimizer = optim.Adam(standard_model.parameters(), lr=learning_rate)

    print("Training Standard Transformer")
    start_time = time.time()
    train_model(standard_model, train_loader, val_loader, standard_criterion, standard_optimizer, device, num_epochs)
    standard_time = time.time() - start_time
    print(f"Standard Transformer training time: {standard_time:.2f} seconds")

    # Longformer
    longformer_model = TransformerClassifier(d_model, n_heads, d_ff, num_layers, attention_type='longformer', window_size=window_size).to(device)
    longformer_criterion = nn.BCEWithLogitsLoss()
    longformer_optimizer = optim.Adam(longformer_model.parameters(), lr=learning_rate)

    print("\nTraining Longformer")
    start_time = time.time()
    train_model(longformer_model, train_loader, val_loader, longformer_criterion, longformer_optimizer, device, num_epochs)
    longformer_time = time.time() - start_time
    print(f"Longformer training time: {longformer_time:.2f} seconds")

    print(f"\nSpeed improvement: {standard_time / longformer_time:.2f}x")

if __name__ == "__main__":
    main()

Training Standard Transformer
Epoch 1/10, Train Loss: 0.2519, Val Loss: 0.0939, Val Acc: 0.9670
Epoch 2/10, Train Loss: 0.0748, Val Loss: 0.0912, Val Acc: 0.9640
Epoch 3/10, Train Loss: 0.0586, Val Loss: 0.0629, Val Acc: 0.9730
Epoch 4/10, Train Loss: 0.0496, Val Loss: 0.0631, Val Acc: 0.9705
Epoch 5/10, Train Loss: 0.0405, Val Loss: 0.0631, Val Acc: 0.9740
Epoch 6/10, Train Loss: 0.0358, Val Loss: 0.1012, Val Acc: 0.9660
Epoch 7/10, Train Loss: 0.0329, Val Loss: 0.0838, Val Acc: 0.9705
Epoch 8/10, Train Loss: 0.0285, Val Loss: 0.0904, Val Acc: 0.9660
Epoch 9/10, Train Loss: 0.0272, Val Loss: 0.1082, Val Acc: 0.9640
Epoch 10/10, Train Loss: 0.0320, Val Loss: 0.0925, Val Acc: 0.9625
Standard Transformer training time: 4555.92 seconds

Training Longformer
Epoch 1/10, Train Loss: 0.4129, Val Loss: 0.2219, Val Acc: 0.9165
Epoch 2/10, Train Loss: 0.2052, Val Loss: 0.1766, Val Acc: 0.9255
Epoch 3/10, Train Loss: 0.1561, Val Loss: 0.1727, Val Acc: 0.9225


KeyboardInterrupt: 

# Clarification on the Longformer Implementation

The provided implementation is not a full, authentic Longformer as described in the original paper by Beltagy et al. Instead, it's a simplified approximation that demonstrates one key aspect of the Longformer: the sliding window attention mechanism.

## Key Differences from True Longformer

1. **Attention Mechanism**: The true Longformer combines sliding window attention with global attention on specific tokens. This implementation only includes the sliding window part.

2. **Efficiency**: The actual Longformer uses specialized CUDA kernels for efficient implementation. This code is a naive Python implementation that doesn't capture the full computational benefits of the Longformer.

3. **Scale**: Longformers are designed to handle much longer sequences (thousands or tens of thousands of tokens) than what this simple implementation can efficiently manage.

4. **Global Attention**: The true Longformer allows for global attention on certain tokens, which this implementation doesn't include.

5. **Dilated Sliding Window**: The actual Longformer can use a dilated sliding window for even longer-range dependencies, which this implementation doesn't have.

## Accurate Description

What's provided is more accurately described as a "sliding window attention mechanism inspired by Longformer". It demonstrates the concept of limiting attention to a local window, but it doesn't capture the full complexity and efficiency of the true Longformer.

For an authentic Longformer implementation, it's recommended to use the official implementation provided by the authors or a well-maintained library like Hugging Face Transformers.